# Topic: File Modes - Access mode matrix, exclusive creation (x), and read/write updates (r+, w+, a+)
 
## 1. THE ACCESS MODE MATRIX
| Mode | Description | Write Action | Read Action | File Pointer Start | Truncates Content? | Creates if Missing? |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
| **`r`** | Read-Only (Default) | No | Yes | Start (0) | No | No (Raises FileNotFoundError) |
| **`w`** | Write-Only | Yes | No | Start (0) | Yes (Clears file!) | Yes |
| **`a`** | Append-Only | Yes | No | End | No | Yes |
| **`x`** | Exclusive Write | Yes | No | Start (0) | No | Yes (Raises FileExistsError if exists) |
| **`r+`**| Read & Write | Yes | Yes | Start (0) | No | No |
| **`w+`**| Write & Read | Yes | Yes | Start (0) | Yes (Clears file!) | Yes |
| **`a+`**| Append & Read | Yes | Yes | End | No | Yes |
 
## 2. EXCLUSIVE CREATION: x MODE
- **Purpose**: Acts as a safety mechanism to prevent overwriting important files.
- **Behavior**: Attempts to create the file. If the file already exists at that path, it raises a `FileExistsError` instead of truncating or opening it.
 
## 3. THE PLUS (+) UPDATING MODES
- **`r+` vs `w+`**:
  - Both allow reading and writing.
  - **`r+`**: Does NOT truncate. Writing at the start overwrites existing characters byte-by-byte.
  - **`w+`**: **Immediately truncates (deletes) all content** to 0 bytes upon opening!
- **`a+`**: Reading can occur anywhere (using `seek`), but writing is always forced to the end of the file.
 
---


In [ ]:
import os

demo_mode_file = "mode_demo.txt"

# 1. Exclusive Creation ('x' mode)
print("--- Exclusive Creation ('x' mode) ---")
try:
    with open(demo_mode_file, "x", encoding="utf-8") as f:
        f.write("Created exclusively.\n")
    print(" -> File created successfully.")
except FileExistsError as e:
    print(f" -> FileExistsError (expected if runs twice): {e}")

# Second attempt must fail since it already exists!
try:
    with open(demo_mode_file, "x", encoding="utf-8") as f:
        f.write("Second write.")
except FileExistsError as e:
    print(f" -> Second write correctly failed: {e}")



In [ ]:
# 2. Read/Write Update ('r+' vs 'w+')
print("\n--- Read/Write Updates ('r+') ---")
# Current contents: "Created exclusively.\n"
with open(demo_mode_file, "r+", encoding="utf-8") as f:
    print(f"Initial read: {repr(f.readline().strip())}")
    f.seek(0)
    # Overwrite the start: "Created" -> "OVERWRT"
    f.write("OVERWRT")
    
with open(demo_mode_file, "r", encoding="utf-8") as f:
    print(f"File contents after r+ overwrite: {repr(f.read())}")
    # Expected: 'OVERWRT exclusively.\n' (Overwrote only matching character offsets)



In [ ]:
print("\n--- Read/Write Updates ('w+') ---")
# opening in 'w+' will immediately delete all content!
with open(demo_mode_file, "w+", encoding="utf-8") as f:
    print(f"Size immediately after opening w+: {f.tell()} bytes")  # Expected: 0
    f.write("Fresh content in w+")
    f.seek(0)
    print(f"Reading from w+: {repr(f.read())}")



In [ ]:
# 3. Clean up
if os.path.exists(demo_mode_file):
    os.remove(demo_mode_file)
